<a href="https://colab.research.google.com/github/Jumpr15/pytorch-work/blob/main/flappy_bird_reinforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flappy-bird-gymnasium

In [135]:
import torch.distributions as distributions
import torch

class Policy:
  def __init__(self, model, optimizer):
    self.model = model
    self.optimizer = optimizer

  def sample_action(self, obs):
    probs = model(obs)
    distri = distributions.Categorical(probs=probs)
    action = distri.sample()
    return action

  def discount_rewards(self, reward_tensor, discount_factor=0.9):
    discount_list = []
    for idx, value in enumerate(reversed(reward_tensor)):
        discount_val = value * (discount_factor ** idx)
        discount_list.append(discount_val)

    discounted_rewards = torch.tensor(discount_list)
    return discounted_rewards

  def calculate_loss(self, train_states, train_actions, train_rewards):
    train_states = torch.tensor(train_states, dtype=torch.float32)
    train_actions = torch.tensor(train_actions, dtype=torch.float32)
    train_rewards = torch.tensor(train_rewards, dtype=torch.float32)

    discounted_rewards = self.discount_rewards(train_rewards)

    probs = policy.model(train_states)
    distri = distributions.Categorical(probs=probs)
    loss = -distri.log_prob(train_actions) * discounted_rewards
    return loss

  def optimize_policy(self, train_states, train_actions, train_rewards):
    loss_tensor = self.calculate_loss(train_states, train_actions, train_rewards)
    loss = loss_tensor.sum()

    self.optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss



In [136]:
import torch.nn as nn
import torch.optim as optim

in_dims = 180
h_dims = 720
out_dims = 2
model = nn.Sequential(
    nn.Linear(in_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, h_dims),
    nn.ReLU(),
    nn.Linear(h_dims, out_dims),
    nn.Sigmoid()
)

lr = 1e-4
optimizer = optim.Adam(
    model.parameters(),
    lr=lr
)

policy = Policy(model, optimizer)

In [ ]:
# @title
import flappy_bird_gymnasium
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

def record_step_trigger(step: int) -> bool:
  per_step = 20
  if step % per_step == 0:
    return True
  return False

env = RecordVideo(gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=True), step_trigger=record_step_trigger, video_folder='./video-output')

In [137]:
def train_epoch(env):
  obs, _ = env.reset()

  train_states = []
  train_actions = []
  train_rewards = []

  while True:
    obs_tensor = (torch.from_numpy(obs)).to(torch.float32)
    action = policy.sample_action(obs_tensor)

    train_states.append(obs)
    train_actions.append(action)

    obs, reward, terminated, _, info = env.step(action)

    train_rewards.append(reward)

    if terminated:
      break

  env.close()
  return train_states, train_actions, train_rewards

In [ ]:
episodes = 200
for _ in range(episodes):
  train_states, train_actions, train_rewards = train_epoch(env)
  loss = policy.optimize_policy(train_states, train_actions, train_rewards)
  print(loss)